# Dataset Exploration

This notebook explores the three main datasets used for protein LLM post-training:

1. **IPD PDB Sample** - Protein structures in PyTorch format
2. **Swiss-Prot** - Curated protein sequences (FASTA)
3. **Mol-Instructions** - Instruction-following pairs for protein tasks

In [ ]:
import os
import json
import gzip
from pathlib import Path
import torch
import pandas as pd
from collections import Counter

# Set data directory
DATA_DIR = Path("../data/raw")
print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Contents: {list(DATA_DIR.iterdir())}")

---
## 1. IPD PDB Sample Dataset

Pre-processed protein structures from the Protein Data Bank in PyTorch tensor format.
Used by RoseTTAFold/ProteinMPNN for structure-based tasks.

In [ ]:
pdb_dir = DATA_DIR / "pdb_2021aug02_sample"
print(f"IPD PDB Sample directory: {pdb_dir}")
print(f"\nDirectory contents:")
for item in pdb_dir.iterdir():
    if item.is_file():
        print(f"  {item.name} ({item.stat().st_size / 1024 / 1024:.2f} MB)")
    else:
        print(f"  {item.name}/")

In [ ]:
# Read the README
readme_path = pdb_dir / "README"
if readme_path.exists():
    print("=" * 60)
    print("README Contents:")
    print("=" * 60)
    print(readme_path.read_text())

In [ ]:
# Count .pt files
pt_files = list(pdb_dir.glob("**/*.pt"))
print(f"Total .pt files: {len(pt_files)}")
print(f"\nFirst 10 files:")
for f in pt_files[:10]:
    print(f"  {f.name}")

In [ ]:
# Load and inspect a sample .pt file
sample_pt = pt_files[0]
print(f"Loading sample file: {sample_pt.name}")
print("=" * 60)

data = torch.load(sample_pt, map_location='cpu', weights_only=False)
print(f"Type: {type(data)}")

if isinstance(data, dict):
    print(f"\nKeys: {list(data.keys())}")
    print("\nContents:")
    for key, value in data.items():
        if isinstance(value, torch.Tensor):
            print(f"  {key}: Tensor shape={value.shape}, dtype={value.dtype}")
        elif isinstance(value, str):
            print(f"  {key}: '{value[:100]}...'" if len(value) > 100 else f"  {key}: '{value}'")
        else:
            print(f"  {key}: {type(value).__name__}")
else:
    print(f"Data: {data}")

In [ ]:
# Load the list.csv for metadata
list_csv = pdb_dir / "list.csv"
if list_csv.exists():
    print("Loading list.csv (first 1000 rows)...")
    df = pd.read_csv(list_csv, nrows=1000)
    print(f"\nShape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print("\nFirst 5 entries:")
    display(df.head())
    print("\nDataset statistics:")
    display(df.describe())

---
## 2. Swiss-Prot Dataset

Curated protein sequences from UniProt in FASTA format.
Contains ~570K reviewed protein sequences with annotations.

In [ ]:
swissprot_dir = DATA_DIR / "swissprot"
fasta_gz = swissprot_dir / "uniprot_sprot.fasta.gz"

print(f"Swiss-Prot file: {fasta_gz}")
print(f"File size: {fasta_gz.stat().st_size / 1024 / 1024:.2f} MB (compressed)")

In [ ]:
def parse_fasta_gz(filepath, max_entries=1000):
    """Parse gzipped FASTA file and return list of (header, sequence) tuples."""
    entries = []
    with gzip.open(filepath, 'rt') as f:
        header = None
        seq_lines = []
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if header is not None:
                    entries.append((header, ''.join(seq_lines)))
                    if len(entries) >= max_entries:
                        break
                header = line[1:]  # Remove '>'
                seq_lines = []
            else:
                seq_lines.append(line)
        if header is not None and len(entries) < max_entries:
            entries.append((header, ''.join(seq_lines)))
    return entries

print("Parsing Swiss-Prot FASTA (first 1000 entries)...")
swissprot_entries = parse_fasta_gz(fasta_gz, max_entries=1000)
print(f"Loaded {len(swissprot_entries)} entries")

In [ ]:
# Display sample entries
print("Sample Swiss-Prot entries:")
print("=" * 60)
for i, (header, seq) in enumerate(swissprot_entries[:5]):
    print(f"\n[Entry {i+1}]")
    print(f"Header: {header[:100]}..." if len(header) > 100 else f"Header: {header}")
    print(f"Sequence length: {len(seq)} amino acids")
    print(f"Sequence (first 80 aa): {seq[:80]}...")

In [ ]:
# Sequence length distribution
seq_lengths = [len(seq) for _, seq in swissprot_entries]
print("Sequence Length Statistics (first 1000 entries):")
print(f"  Min: {min(seq_lengths)}")
print(f"  Max: {max(seq_lengths)}")
print(f"  Mean: {sum(seq_lengths)/len(seq_lengths):.1f}")
print(f"  Median: {sorted(seq_lengths)[len(seq_lengths)//2]}")

# Amino acid composition
all_aa = ''.join(seq for _, seq in swissprot_entries)
aa_counts = Counter(all_aa)
print(f"\nAmino Acid Composition:")
for aa, count in aa_counts.most_common(10):
    print(f"  {aa}: {count:,} ({100*count/len(all_aa):.2f}%)")

In [ ]:
# Parse header for species and protein info
def parse_swissprot_header(header):
    """Parse Swiss-Prot FASTA header.
    Format: sp|ACCESSION|ENTRY_NAME PROTEIN_NAME OS=ORGANISM OX=TAXID GN=GENE PE=EVIDENCE SV=VERSION
    """
    parts = header.split(' ', 1)
    ids = parts[0].split('|')
    accession = ids[1] if len(ids) > 1 else 'N/A'
    entry_name = ids[2] if len(ids) > 2 else 'N/A'
    
    description = parts[1] if len(parts) > 1 else ''
    
    # Extract organism
    organism = 'N/A'
    if 'OS=' in description:
        start = description.index('OS=') + 3
        end = description.find(' OX=', start) if ' OX=' in description else len(description)
        organism = description[start:end]
    
    return {'accession': accession, 'entry_name': entry_name, 'organism': organism}

# Create DataFrame
swissprot_data = []
for header, seq in swissprot_entries:
    info = parse_swissprot_header(header)
    info['sequence_length'] = len(seq)
    info['sequence'] = seq[:50] + '...' if len(seq) > 50 else seq
    swissprot_data.append(info)

swissprot_df = pd.DataFrame(swissprot_data)
print("Swiss-Prot entries as DataFrame:")
display(swissprot_df.head(10))

In [ ]:
# Top organisms
print("Top 10 Organisms:")
print(swissprot_df['organism'].value_counts().head(10))

---
## 3. Mol-Instructions Dataset

Instruction-following pairs for protein-related tasks.
Contains multiple task types: protein design, function prediction, catalytic activity, etc.

In [ ]:
mol_inst_dir = DATA_DIR / "mol_instructions" / "data" / "Protein-oriented_Instructions"
print(f"Mol-Instructions directory: {mol_inst_dir}")
print("\nAvailable files:")

json_files = list(mol_inst_dir.glob("*.json"))
for f in json_files:
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name}: {size_mb:.2f} MB")

In [ ]:
def load_json_samples(filepath, max_samples=100):
    """Load samples from a JSON file."""
    with open(filepath, 'r') as f:
        data = json.load(f)
    if isinstance(data, list):
        return data[:max_samples]
    return data

# Load samples from each file
mol_instructions_data = {}
for json_file in json_files:
    task_name = json_file.stem
    print(f"Loading {task_name}...")
    samples = load_json_samples(json_file, max_samples=100)
    mol_instructions_data[task_name] = samples
    print(f"  Loaded {len(samples)} samples")

In [ ]:
# Explore each task type
for task_name, samples in mol_instructions_data.items():
    print("=" * 60)
    print(f"Task: {task_name}")
    print("=" * 60)
    
    if samples:
        sample = samples[0]
        print(f"Fields: {list(sample.keys())}")
        print(f"\nSample entry:")
        for key, value in sample.items():
            if isinstance(value, str):
                display_val = value[:200] + '...' if len(value) > 200 else value
                print(f"  {key}: {display_val}")
            else:
                print(f"  {key}: {value}")
    print()

In [ ]:
# Show more detailed examples for each task
print("\n" + "=" * 80)
print("DETAILED EXAMPLES FROM EACH TASK")
print("=" * 80)

for task_name, samples in mol_instructions_data.items():
    print(f"\n{'='*60}")
    print(f"TASK: {task_name.upper()}")
    print(f"{'='*60}")
    
    for i, sample in enumerate(samples[:3]):
        print(f"\n--- Example {i+1} ---")
        if 'instruction' in sample:
            print(f"INSTRUCTION: {sample['instruction'][:300]}..." if len(sample.get('instruction', '')) > 300 else f"INSTRUCTION: {sample.get('instruction', 'N/A')}")
        if 'input' in sample:
            input_text = sample['input']
            print(f"INPUT: {input_text[:300]}..." if len(input_text) > 300 else f"INPUT: {input_text}")
        if 'output' in sample:
            output_text = sample['output']
            print(f"OUTPUT: {output_text[:300]}..." if len(output_text) > 300 else f"OUTPUT: {output_text}")

In [ ]:
# Summary statistics
print("\n" + "=" * 60)
print("MOL-INSTRUCTIONS SUMMARY")
print("=" * 60)

for task_name, samples in mol_instructions_data.items():
    # Get actual file size to estimate total entries
    json_file = mol_inst_dir / f"{task_name}.json"
    with open(json_file, 'r') as f:
        full_data = json.load(f)
    
    total_entries = len(full_data) if isinstance(full_data, list) else 1
    
    # Calculate average lengths
    if samples and 'input' in samples[0]:
        avg_input_len = sum(len(s.get('input', '')) for s in samples) / len(samples)
        avg_output_len = sum(len(s.get('output', '')) for s in samples) / len(samples)
        print(f"\n{task_name}:")
        print(f"  Total entries: {total_entries:,}")
        print(f"  Avg input length: {avg_input_len:.0f} chars")
        print(f"  Avg output length: {avg_output_len:.0f} chars")

---
## Summary

### Dataset Overview

| Dataset | Location | Format | Size |
|---------|----------|--------|------|
| IPD PDB Sample | `data/raw/pdb_2021aug02_sample` | PyTorch .pt files | ~870 structures |
| Swiss-Prot | `data/raw/swissprot/uniprot_sprot.fasta.gz` | Gzipped FASTA | ~570K sequences |
| Mol-Instructions | `data/raw/mol_instructions/data/Protein-oriented_Instructions/` | JSON files | ~500K instruction pairs |

In [ ]:
print("Dataset exploration complete!")